# 04 — Origem: JSON

Le JSON da zona de entrada e grava na camada `coleta`.

O Spark espera **JSON Lines** por padrao: um objeto por linha, sem virgula entre eles.

```
{"id": 1, "uf": "PE"}
{"id": 2, "uf": "SP"}
```

Para um array unico dentro do arquivo (`[{...}, {...}]`), passe `multiLine=True`.

In [ ]:
from lakehouse import sessao, ler_arquivo, gravar, perfil

spark = sessao("04-json")

In [ ]:
ARQUIVO = "pedidos.json"
DESTINO = "coleta.pedidos"

df = ler_arquivo(spark, ARQUIVO)
perfil(df)

## JSON aninhado

Estrutura hierarquica nao vai bem numa tabela analitica. Achate antes de gravar.

`.` acessa campo de struct; `explode` transforma cada item de uma lista em uma linha.

```python
from pyspark.sql import functions as F

# {"id":1, "cliente":{"nome":"Ana","uf":"PE"}, "itens":[{"sku":"A1","qtd":2}]}
plano = (df
    .select("id",
            F.col("cliente.nome").alias("cliente_nome"),
            F.col("cliente.uf").alias("cliente_uf"),
            F.explode("itens").alias("item"))
    .select("*", F.col("item.sku").alias("sku"), F.col("item.qtd").alias("qtd"))
    .drop("item"))
```

Para inspecionar a estrutura antes: `df.printSchema()`.

In [ ]:
df.printSchema()

## Outras formas de JSON

**Multiline (array unico):**
```python
df = ler_arquivo(spark, "dados.json", multiLine=True)
```

**Resposta de API REST** — sem arquivo intermediario:
```python
import requests, json
dados = requests.get("https://api.exemplo.com/vendas", timeout=30).json()
df = spark.read.json(spark.sparkContext.parallelize([json.dumps(r) for r in dados]))
```
Serve para volumes pequenos e medios; a leitura acontece no driver.

**Schema divergente entre arquivos** — o Spark une os campos e preenche o que falta com nulo. Para
detectar registros fora do padrao, leia com `mode="PERMISSIVE"` e
`columnNameOfCorruptRecord="_corrompido"`, depois filtre essa coluna.

In [ ]:
gravar(df, DESTINO, modo="substituir")